In [6]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
advanced_features = [
    "beta_6m",
    "beta_12m",
    "corr_spy_6m",
    "corr_spy_12m",
    "mom_3m_rank_pct",
    "mom_6m_rank_pct",
    "mom_12m_rank_pct",
    "vol_3m_rank_pct",
    "vol_6m_rank_pct",
    "vol_12m_rank_pct",
    "sector_return_dispersion",
]

baseline_numeric_features = [
    "mom_1m",
    "mom_3m",
    "mom_6m",
    "mom_12m",
    "vol_3m",
    "vol_6m",
    "vol_12m",
    "drawdown_12m",

    "spy_mom_1m",
    "spy_mom_3m",
    "spy_mom_6m",
    "spy_mom_12m",
    "spy_vol_3m",
    "spy_vol_6m",
    "spy_vol_12m",
    "spy_drawdown_12m",

    "DGS10",
    "DGS2",
    "DFF",
    "VIXCLS",
    "yield_spread_10y_2y",
    "DGS10_change_1m",
    "DGS2_change_1m",
    "DFF_change_1m",
    "VIXCLS_change_1m",
    "yield_spread_10y_2y_change_1m",
]

In [4]:
advanced_numeric_features = (
    baseline_numeric_features
    + advanced_features
)

In [5]:
print("Baseline numeric features:", len(baseline_numeric_features))
print("Advanced numeric features:", len(advanced_numeric_features))

Baseline numeric features: 26
Advanced numeric features: 37


In [10]:
from pathlib import Path

# Cell 2: project path
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

advanced_df = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "advanced_model_dataset.csv",
    parse_dates=["Date"],
)

advanced_df.shape

(1181, 53)

In [11]:
def evaluate_feature_set(
    df,
    numeric_features,
    feature_set_name,
    validation_years=(2019, 2020, 2021, 2022),
):

    results = []

    for validation_year in validation_years:

        fold_train = df[
            (df["Date"].dt.year >= 2016)
            & (df["Date"].dt.year < validation_year)
        ].copy()

        fold_val = df[
            df["Date"].dt.year == validation_year
        ].copy()

        X_train = fold_train[
            numeric_features + ["Ticker"]
        ]
        y_train = fold_train["target"]

        X_val = fold_val[
            numeric_features + ["Ticker"]
        ]
        y_val = fold_val["target"]

        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    numeric_features,
                ),
                (
                    "ticker",
                    OneHotEncoder(
                        drop="first",
                        handle_unknown="ignore",
                    ),
                    ["Ticker"],
                ),
            ]
        )

        model = LogisticRegression(
            penalty="l2",
            C=1,
            class_weight="balanced",
            max_iter=2000,
            random_state=42,
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model),
        ])

        pipeline.fit(X_train, y_train)

        predictions = pipeline.predict(X_val)
        probabilities = pipeline.predict_proba(X_val)[:, 1]

        results.append({
            "feature_set": feature_set_name,
            "validation_year": validation_year,
            "roc_auc": roc_auc_score(
                y_val,
                probabilities,
            ),
            "balanced_accuracy":
                balanced_accuracy_score(
                    y_val,
                    predictions,
                ),
            "f1": f1_score(
                y_val,
                predictions,
                zero_division=0,
            ),
        })

    return pd.DataFrame(results)

In [12]:
baseline_results = evaluate_feature_set(
    advanced_df,
    baseline_numeric_features,
    "baseline",
)

advanced_results = evaluate_feature_set(
    advanced_df,
    advanced_numeric_features,
    "advanced",
)

feature_comparison = pd.concat(
    [baseline_results, advanced_results],
    ignore_index=True,
)

feature_comparison

/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/sector-etf-ml/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/

,feature_set,validation_year,roc_auc,balanced_accuracy,f1
0,baseline,2019,0.543052,0.537084,0.200000
1,baseline,2020,0.499719,0.489899,0.411765
2,baseline,2021,0.516569,0.478697,0.415094
3,baseline,2022,0.439429,0.440000,0.375000
4,advanced,2019,0.547315,0.555413,0.395062
5,advanced,2020,0.513749,0.483165,0.522388
6,advanced,2021,0.502089,0.461571,0.258824
7,advanced,2022,0.454571,0.455714,0.355140


In [13]:
feature_comparison.groupby(
    "feature_set"
).agg(
    mean_roc_auc=("roc_auc", "mean"),
    std_roc_auc=("roc_auc", "std"),
    mean_balanced_accuracy=(
        "balanced_accuracy",
        "mean",
    ),
    mean_f1=("f1", "mean"),
)

,mean_roc_auc,std_roc_auc,mean_balanced_accuracy,mean_f1
feature_set,,,,
advanced,0.504431,0.038372,0.488966,0.382853
baseline,0.499692,0.043957,0.486420,0.350465
